### Calibrate a functional dataset into clinical evidence strengths

This notebook takes a **functional assay score** (or any in-silico predictor), aligns it to ClinVar's
clinical truth set, and runs the ClinGen-SVI local-calibration method
([Pejaver et al. 2022](https://doi.org/10.1016/j.ajhg.2022.10.013)) to convert raw scores into
ACMG/AMP evidence strengths (PP3 / BP4 at Supporting → Very Strong).

We use **AlphaMissense** scores for LDLR as the worked example because they are public — substitute your
own functional data by editing the **Parameters** cell. Everything downstream is parameterized.

**Pipeline:** load truth set → load + merge functional scores → inspect P/B separation → set/estimate the
class prior `alpha` → run calibration + inference → attach ACMG strengths and summarize.

> Run `01_process_clinvar.ipynb` first to produce the ClinVar table this notebook consumes.

### Setup: install the calibration tools

Two external repos are needed. Clone them into the repo root (or anywhere on your path):

```bash
# Required: the ClinGen-SVI local calibration code
git clone https://github.com/pejaverlab/clingen-svi-comp_calibration.git

# Optional: only needed if you want to ESTIMATE alpha (the class prior) from the data
git clone https://github.com/Dzeiberg/dist_curve.git
pip install -e dist_curve
```

- `clingen-svi-comp_calibration` is run as a script (`main.py calibrate` / `main.py infer`) and ships a
  `config.ini` template — point `CALIB_DIR` (below) at its folder.
- `dist_curve` is imported as a Python package and additionally needs a trained estimator weights file
  (`model.hdf5`, ~11 MB) from that project; point `ESTIMATOR_MODEL_PATH` at it. Skip both if you supply `alpha` yourself.

Python deps: `pandas numpy scipy plotnine` (+ `tensorflow` only for alpha estimation).

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import configparser

import pandas as pd
import numpy as np
from plotnine import *
from scipy.stats import ks_2samp


In [ ]:
# Resolve the repo root whether run from the repo root or from notebooks/
proj_dir = Path.cwd()
if proj_dir.name == 'notebooks':
    proj_dir = proj_dir.parent

data_dir = proj_dir / 'data'
clinvar_dir = data_dir / 'clinvar'
example_dir = data_dir / 'example'
results_dir = proj_dir / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

# Cloned calibration tools (edit if you cloned them elsewhere)
CALIB_DIR = proj_dir / 'clingen-svi-comp_calibration'   # required: has main.py + config.ini
ESTIMATOR_MODEL_PATH = proj_dir / 'model.hdf5'          # optional: dist_curve estimator weights


### Parameters

This is the **only cell most users need to edit.** Define the gene, where the functional scores live,
which column(s) hold the score, and how to calibrate.

- `ALPHA` — the class prior (fraction of pathogenic variants among all assayed). Provide a value, **or**
  set `ESTIMATE_ALPHA = True` to estimate it from the data with `dist_curve`.
- `GAUSSIAN_SMOOTHING` — optional smoothing of the calibration curve.
- Higher scores are assumed to indicate **more pathogenic**. If your assay is reversed, negate the column first.

In [ ]:
GENE = 'LDLR'

# Functional data: a delimited file with columns identifying the variant + score(s).
FUNC_DATA_PATH = example_dir / 'LDLR_AlphaMissense_aa_substitutions.tsv'
FILE_PREFIX = 'LDLR_am'                 # used to name output files/dirs
SCORE_COLS = ['am_score']               # one or more score columns to calibrate
SCORE_LABELS = ['AlphaMissense score']  # human-readable labels, aligned with SCORE_COLS

# Calibration settings
ALPHA = 0.201               # class prior; ignored if ESTIMATE_ALPHA is True
ESTIMATE_ALPHA = False      # True -> estimate ALPHA per score column via dist_curve
GAUSSIAN_SMOOTHING = False  # True -> smooth the calibration curve

# Where calibration/inference outputs are written
inference_dir = results_dir / 'example_output'
inference_dir.mkdir(parents=True, exist_ok=True)

base_cols = ['gene', 'aapos', 'aaref', 'aaalt']


### Load the ClinVar truth set and resolve one label per substitution

Collapse multiple ClinVar rows for the same amino-acid substitution to a single `cs_simple`.
If both Pathogenic and Benign assertions exist for a substitution, Pathogenic wins; synonymous changes
(`aaref == aaalt`) are treated as Benign.

In [ ]:
def resolve_cs_simple(row):
    if row['aaref'] == row['aaalt']:
        return 'b'
    unique = set(row['cs_simple_unique_list'])
    if 'p' in unique:
        return 'p'
    if 'b' in unique:
        return 'b'
    return pd.NA

# Pick the dated CSV from notebook 1; fall back to the bundled LDLR example subset.
candidates = sorted(clinvar_dir.glob(f'clinvar_coding_GRCh38_*_{GENE}.csv.gz')) \
           + sorted(clinvar_dir.glob('clinvar_coding_GRCh38_*.csv.gz'))
clinvar_path = candidates[0]
print('Using ClinVar table:', clinvar_path.name)

df_clinVar = pd.read_csv(clinvar_path)
df_clinVar = df_clinVar.loc[(df_clinVar['gene'] == GENE) & (df_clinVar['cs_simple'].isin(['p', 'b']))]

df_clinVar_lite = (
    df_clinVar.groupby(['gene', 'aapos', 'aaref', 'aaalt'])
    .agg(cs_simple_unique_list=('cs_simple', lambda x: list(set(x))))
    .reset_index()
)
df_clinVar_lite['cs_simple_resolved'] = df_clinVar_lite.apply(resolve_cs_simple, axis=1)
print('unique labelled substitutions:', len(df_clinVar_lite))
df_clinVar_lite.head()


### Load functional scores and merge clinical labels

Parse the variant identity into `aapos/aaref/aaalt`, then left-join the resolved ClinVar label.

**Adapt this cell to your file's format.** The AlphaMissense TSV has no header and encodes the substitution
in an `aa_str` like `M1A`. Your file may already have `aapos/aaref/aaalt` columns, in which case just read it
and ensure a `gene` column exists.

In [ ]:
# --- Example parser for the AlphaMissense TSV (uniprot_id, aa_str, am_score, am_class) ---
df_scores = pd.read_csv(FUNC_DATA_PATH, sep='\t',
                        names=['uniprot_id', 'aa_str', 'am_score', 'am_class'])
df_scores['gene'] = GENE
df_scores['aapos'] = df_scores['aa_str'].str[1:-1].astype(int)
df_scores['aaref'] = df_scores['aa_str'].str[0]
df_scores['aaalt'] = df_scores['aa_str'].str[-1]
# -----------------------------------------------------------------------------------------

df_scores = df_scores.merge(
    df_clinVar_lite[['gene', 'aapos', 'aaref', 'aaalt', 'cs_simple_resolved']],
    on=['gene', 'aapos', 'aaref', 'aaalt'], how='left',
).rename(columns={'cs_simple_resolved': 'cs_simple'})
df_scores = df_scores.drop_duplicates(ignore_index=True)

for score_col, label in zip(SCORE_COLS, SCORE_LABELS):
    d = df_scores.loc[df_scores[score_col].notna()]
    print(f"{label}: {len(d)} scored | P/LP={ (d['cs_simple']=='p').sum() } | B/LB={ (d['cs_simple']=='b').sum() }")


### Inspect class separation (density + KS test)

Before calibrating, sanity-check that Pathogenic and Benign variants actually separate on the score.
A small Kolmogorov–Smirnov p-value and visibly shifted densities indicate the assay carries clinical signal.

In [ ]:
for score_col, score_label in zip(SCORE_COLS, SCORE_LABELS):
    df_plt = df_scores.loc[df_scores['cs_simple'].isin(['p', 'b']), base_cols + [score_col]].copy()
    df_plt = df_plt.rename(columns={score_col: 'score_value'})
    df_plt = df_plt.loc[df_plt['score_value'].notna()]
    df_plt['cs_label'] = df_plt['cs_simple'].map({'p': 'Pathogenic', 'b': 'Benign'})

    ks_p = ks_2samp(df_plt.loc[df_plt.cs_label == 'Pathogenic', 'score_value'],
                    df_plt.loc[df_plt.cs_label == 'Benign', 'score_value']).pvalue
    ks_lab = pd.DataFrame({'label': [f'KS p = {ks_p:.2e}']})

    p = (ggplot(df_plt, aes(x='score_value', fill='cs_label'))
         + geom_density(alpha=0.5)
         + scale_fill_manual(values=['blue', 'red'])
         + geom_text(ks_lab, aes(label='label'), inherit_aes=False,
                     x=np.inf, y=np.inf, ha='right', va='top', size=10)
         + labs(title=score_label, x='Score value', y='Density', fill='ClinVar')
         + theme_light() + theme(figure_size=(6, 4)))
    p.show()


### (Optional) Estimate the class prior `alpha` with `dist_curve`

The calibration needs `alpha`, the fraction of pathogenic variants among **all** assayed variants.
If you don't have a principled value, `dist_curve` estimates it by treating Pathogenic variants as the
'positive' component and all scored variants as the 'mixture'. Requires `dist_curve` installed and the
estimator weights at `ESTIMATOR_MODEL_PATH`. This cell is a no-op unless `ESTIMATE_ALPHA = True`.

In [ ]:
def estimate_alpha(df, score_col):
    """Estimate the class prior for one score column using dist_curve."""
    from dist_curve.curve_constructor import makeCurve, plotCurve
    from dist_curve.model import getTrainedEstimator

    pos = df.loc[df['cs_simple'] == 'p', score_col].dropna().to_numpy().reshape(-1, 1)
    mix = df[score_col].dropna().to_numpy().reshape(-1, 1)
    curve = makeCurve(pos, mix)
    print(plotCurve(curve))

    model = getTrainedEstimator(str(ESTIMATOR_MODEL_PATH))
    a = float(model.predict(curve.reshape((1, -1)) / curve.sum())[0][0])
    return a

# Quick smoke test (only runs when requested)
if ESTIMATE_ALPHA:
    print('alpha estimate for', SCORE_COLS[0], '=', estimate_alpha(df_scores, SCORE_COLS[0]))
else:
    print(f'Using predefined ALPHA = {ALPHA} (set ESTIMATE_ALPHA = True to estimate per score).')


### Calibrate and infer ACMG evidence strengths

For each score column this:
1. picks `alpha` (estimated or predefined),
2. writes the calibration `config.ini` and the labelled / unlabelled score files,
3. runs `main.py calibrate` then `main.py infer` from the cloned tool,
4. joins the inferred `ACMG20` / `ACMG18` strengths back onto every variant and saves a CSV.

`ACMG18`/`ACMG20` are the assigned evidence strengths (e.g. *Pathogenic Strong*, *Benign Supporting*,
*Variant of Uncertain Significance*) — the calibrated, clinically interpretable output.

In [ ]:
calib_main = CALIB_DIR / 'main.py'
config_template = CALIB_DIR / 'config.ini'
assert calib_main.exists(), f'Clone clingen-svi-comp_calibration to {CALIB_DIR} first'

for score_col in SCORE_COLS:
    smoothing_tag = 'gaussian_smoothing' if GAUSSIAN_SMOOTHING else 'no_smoothing'
    run_name = f'{FILE_PREFIX}__{score_col}__calibrated_all__{smoothing_tag}'
    out_dir = inference_dir / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) class prior
    alpha = estimate_alpha(df_scores, score_col) if ESTIMATE_ALPHA else ALPHA
    print(f'{run_name}: alpha = {alpha}')

    # 2) config
    config = configparser.ConfigParser()
    config.read(config_template)
    config['priorinfo']['alpha'] = str(alpha)
    if GAUSSIAN_SMOOTHING:
        config['smoothing']['gaussian_smoothing'] = 'True'
    config_file = out_dir / 'calibration_config.ini'
    with open(config_file, 'w') as fh:
        config.write(fh)

    # labelled data: score + binary label (1=pathogenic, 0=benign)
    df_lab = df_scores.loc[df_scores['cs_simple'].isin(['p', 'b']), [score_col, 'cs_simple']].dropna().copy()
    df_lab['label'] = (df_lab['cs_simple'] == 'p').astype(int)
    df_lab[[score_col, 'label']].to_csv(out_dir / 'labelled_data.txt', index=False, header=False, sep='\t')

    # unlabelled data: every scored variant
    df_unlab = df_scores.loc[df_scores[score_col].notna(), base_cols + [score_col]].reset_index(drop=True)
    df_unlab[[score_col]].to_csv(out_dir / 'unlabelled_data.txt', index=False, header=False, sep='\t')

    # 3) calibrate + infer
    subprocess.run([sys.executable, str(calib_main), 'calibrate',
                    '--configfile', str(config_file),
                    '--labelled_data_file', str(out_dir / 'labelled_data.txt'),
                    '--unlabelled_data_file', str(out_dir / 'unlabelled_data.txt'),
                    '--outdir', str(out_dir)], check=True)
    # run infer from inside out_dir so infer_out.txt lands there (imports still resolve
    # from main.py's own directory regardless of cwd)
    subprocess.run([sys.executable, str(calib_main), 'infer',
                    '--score_file', str(out_dir / 'unlabelled_data.txt'),
                    '--calibrated_data_directory', str(out_dir)], check=True, cwd=str(out_dir))

    # 4) join inferred strengths back onto the variants
    df_infer = pd.read_csv(out_dir / 'infer_out.txt', sep='\t', skiprows=12)
    df_infer = pd.concat([df_unlab, df_infer[['ACMG20', 'ACMG18']]], axis=1)
    out_csv = inference_dir / f'{run_name}.infer_out.csv'
    df_infer.to_csv(out_csv, index=False)
    print(f'{run_name}: wrote {out_csv.name}')


### Summarize: calibrated strengths vs ClinVar

Cross-tabulate the inferred ACMG strength against the known ClinVar labels for the variants that have
both, as a final check that pathogenic strengths land on Pathogenic variants and vice versa.

In [ ]:
score_col = SCORE_COLS[0]
smoothing_tag = 'gaussian_smoothing' if GAUSSIAN_SMOOTHING else 'no_smoothing'
run_name = f'{FILE_PREFIX}__{score_col}__calibrated_all__{smoothing_tag}'

df_infer = pd.read_csv(inference_dir / f'{run_name}.infer_out.csv')
df_infer = df_infer.merge(
    df_clinVar_lite[['gene', 'aapos', 'aaref', 'aaalt', 'cs_simple_resolved']],
    on=['gene', 'aapos', 'aaref', 'aaalt'], how='left',
).rename(columns={'cs_simple_resolved': 'cs_simple'})

labelled = df_infer.loc[df_infer['cs_simple'].isin(['p', 'b'])]
print('ACMG18 strength vs ClinVar label:')
pd.crosstab(labelled['ACMG18'], labelled['cs_simple'])
